# Week 3: Exploratory Data Analysis & Feature Engineering
## Notebook 12: Missingness Mechanisms: MCAR, MAR & MNAR

### 1. Conceptual Intuition & Theoretical Background
Imputing missing values with mean or median without understanding the underlying **missingness mechanism** introduces severe statistical bias.

Donald Rubin's Missing Data Taxonomy:
1. **Missing Completely at Random (MCAR):**
   Missingness probability is independent of both observed and unobserved data:
   $$P(M \mid Y_{\text{obs}}, Y_{\text{mis}}) = P(M)$$
   *Example:* A laboratory tube drops on the floor and breaks randomly.
2. **Missing at Random (MAR):**
   Missingness depends on other observed features, but not on the missing value itself:
   $$P(M \mid Y_{\text{obs}}, Y_{\text{mis}}) = P(M \mid Y_{\text{obs}})$$
   *Example:* Men are less likely to report their weight, but conditional on gender, missingness is random.
3. **Missing Not at Random (MNAR):**
   Missingness depends directly on the unobserved value itself:
   $$P(M \mid Y_{\text{obs}}, Y_{\text{mis}}) \neq P(M \mid Y_{\text{obs}})$$
   *Example:* Individuals with very high wealth refuse to disclose their income on tax surveys.

---

### 2. Scratch Python Implementation: Diagnosing Missingness Dependency

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

# Generate Synthetic Survey Dataset (N=1000)
np.random.seed(42)
n = 1000
age = np.random.normal(45, 12, n)
income = 2000.0 * age + np.random.normal(0, 15000, n)

# 1. Synthesize MCAR: 10% completely random dropped entries
mcar_mask = np.random.rand(n) < 0.10
income_mcar = income.copy()
income_mcar[mcar_mask] = np.nan

# 2. Synthesize MAR: Older individuals (>55) are 3x more likely to omit income
prob_mar = np.where(age > 55, 0.35, 0.05)
mar_mask = np.random.rand(n) < prob_mar
income_mar = income.copy()
income_mar[mar_mask] = np.nan

# 3. Diagnostic Test for MAR: T-Test of observed feature 'Age' grouped by Missingness Indicator
def test_mar_dependency(feature_observed, target_missing):
    indicator = np.isnan(target_missing).astype(int)
    group_present = feature_observed[indicator == 0]
    group_missing = feature_observed[indicator == 1]
    
    t_stat, p_val = stats.ttest_ind(group_present, group_missing, equal_var=False)
    return t_stat, p_val, np.mean(group_present), np.mean(group_missing)

t_mcar, p_mcar, mean_pres_mcar, mean_miss_mcar = test_mar_dependency(age, income_mcar)
t_mar, p_mar, mean_pres_mar, mean_miss_mar = test_mar_dependency(age, income_mar)

print("--- Missingness Diagnostic Hypothesis Testing ---")
print(f"MCAR Scenario -> p-value: {p_mcar:.4f} (Fail to reject H0: Age means are identical)")
print(f"  Observed Age Mean: {mean_pres_mcar:.1f} | Missing Age Mean: {mean_miss_mcar:.1f}")

print(f"\nMAR Scenario  -> p-value: {p_mar:.2e} (Reject H0: Statistically significant age bias!)")
print(f"  Observed Age Mean: {mean_pres_mar:.1f} | Missing Age Mean: {mean_miss_mar:.1f}")
print("Significant p-value confirms missingness is conditionally driven by observed variable 'Age'!")

### 3. Step-by-Step Code Walkthrough
1. **Missingness Indicator:** Creates a binary flag vector indicating presence vs absence.
2. **Two-Sample Welch's T-Test:** Tests whether the distribution of an observed auxiliary variable differs between missing and non-missing cohorts.
3. **Statistical Inference:** Low p-value ($p < 0.001$) rules out MCAR and establishes MAR.

---

### 4. Industry Applications & Trade-offs
- **Clinical Trials & Healthcare Records:** Patient drop-out often represents MNAR (patients drop out due to severe side-effects).
- **Trade-off:** Complete Case Analysis (dropping missing rows) produces biased parameter estimates under MAR and MNAR.

---

### 5. Top Interview Questions & Answers
**Q1: Can statistical hypothesis tests prove that a dataset is MNAR?**
> **Answer:** No. By definition, MNAR depends on unobserved values. Proving MNAR requires domain knowledge, follow-up contact with non-responders, or sensitivity modeling (Pattern-Mixture Models).

**Q2: When is Multiple Imputation by Chained Equations (MICE) valid?**
> **Answer:** MICE is mathematically valid under the Missing at Random (MAR) assumption, where missing values can be predicted using observed auxiliary covariates.